# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and define the dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset object, not a dict

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant structure.

### List Record Sets, Fields, and Columns with their `@id`
All entities are referenced by their `@id`. We'll show all record sets, field `@id`s, and column `@id`s.

In [ ]:
# List available record sets and fields by @id

if not hasattr(metadata, 'record_sets'):
    print("No record sets were found in the metadata.")
else:
    print("Available record sets:")
    for record_set in metadata.record_sets:
        print(f"- Record Set Name: {getattr(record_set, 'name', 'N/A')}")
        print(f"  @id: {record_set.id}")
        if hasattr(record_set, 'fields'):
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - Field Name: {getattr(field, 'name', 'N/A')} | @id: {field.id}")
        if hasattr(record_set, 'columns'):
            print("  Columns:")
            for column in record_set.columns:
                print(f"    - Column Name: {getattr(column, 'name', 'N/A')} | @id: {column.id}")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field / column `@id`s from the overview above.

In [ ]:
# Automatically gather record set @id's from the metadata.
if hasattr(metadata, 'record_sets'):
    record_sets_ids = [record_set.id for record_set in metadata.record_sets]
    print("Discovered record set @id's:")
    for rsid in record_sets_ids:
        print("-", rsid)
else:
    record_sets_ids = []
    print("No record sets present.")

# Extract data from each record set by @id using mlcroissant
dataframes = {}
for rsid in record_sets_ids:
    print(f"Loading records for record set @id='{rsid}' ...")
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} rows, columns: {list(df.columns)}\n")

# Preview the first record set if any
if record_sets_ids:
    main_rs_id = record_sets_ids[0]
    print(f"First record set @id: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print("No record sets to preview.")

## 4. Exploratory Data Analysis (EDA)
Perform common data processing operations such as filtering, normalization, and grouping on the loaded DataFrame.

> We'll pick a numeric field for demo (e.g. 'Age'), use its `@id` as shown above. Please adapt field selection as appropriate for your schema.

In [ ]:
# --------- Select a numeric field and a grouping field by @id ---------
# You may need to adapt these IDs according to your own record set/field structure.
if record_sets_ids:
    record_set_id = main_rs_id
    df = dataframes[record_set_id]
    
    # Try to pick likely field IDs for demo; modify as needed for your dataset
    numeric_field_candidates = [col for col in df.columns if 'age' in col.lower()]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Selected numeric field by @id: {numeric_field}")
    else:
        # If no 'age' found, pick first numeric-like column
        numeric_field = None
        for col in df.select_dtypes(include=['number']).columns:
            numeric_field = col
            print(f"Selected first numeric column by @id: {numeric_field}")
            break
        if numeric_field is None:
            print("No numeric field found in record set.")
    
    # Pick a categorical/group field
    group_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'gender', 'msi', 'status', 'location', 'group'])]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Selected group field by @id: {group_field}")
    else:
        group_field = None
        print("No group/categorical field found.")
else:
    numeric_field = None
    group_field = None


# --------- EDA: Filtering, Normalization, Grouping ---------
if numeric_field and record_sets_ids:
    threshold = 50  # Example threshold (e.g., Age>50)
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}: {len(filtered_df)}")

    # Normalize the numeric field (z-score)
    filtered_df = filtered_df.copy() # to avoid SettingWithCopyWarning
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Sample normalized {numeric_field}:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group field and show mean of numeric field
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print('No numeric field available to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset by referencing each with its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and record_sets_ids:
    # Histogram of the numeric field
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
We have demonstrated how to load and explore the FAIR² dataset using `mlcroissant` and referenced all fields and record sets using their `@id`. You can extend this notebook for deeper domain-specific analysis using statistical or machine learning methods as appropriate for the clinical and molecular data.